# 🧭 Intro to Microsoft Foundry — a developer's quick-start guide

> ⏱ ~15 min read · no setup required · the map you keep open while you work through Labs 0 → 2.

Welcome. This notebook is the **one-stop developer guide** for the Cohere-on-Foundry workshop.
It is intentionally light on code and heavy on *why* — so you understand the moving parts before
you start running cells in the labs.

**By the end of this read you will know:**

1. What **Microsoft Foundry** is and what problem it solves for you.
2. Why this workshop uses **Cohere** models on Foundry.
3. How the **Microsoft Agent Framework (MAF)** complements Foundry.
4. The **build → measure → improve** loop that turns a first-draft agent into a production-grade one.
5. Where each lab fits and what you will take away from it.

There are no exercises here. Skim it once, then jump to [Lab 0](./lab-0-setup/SETUP.md).

## 1. The scenario you will build

Picture a corporate **travel concierge**. An employee types one sentence —
*"Plan a multi-leg trip from San Francisco to Seattle to Chicago, plus hotels"* — and the
concierge handles flights, hotels, and the company's travel policy behind the scenes.

```mermaid
flowchart LR
    user["👤 Employee<br/>'Plan my SFO → SEA → CHI trip'"]
    concierge["🧳 Concierge agent<br/>(orchestrator)"]
    flight["✈️ Flight agent"]
    hotel["🏨 Hotel agent"]
    car["🚗 Car agent"]
    catalogs["📚 flights.json<br/>hotels.json<br/>cars.json<br/>travel-policy.md"]
    user --> concierge
    concierge -->|agents as tools| flight
    concierge -->|agents as tools| hotel
    concierge -->|agents as tools| car
    flight --> catalogs
    hotel --> catalogs
    car --> catalogs
```

**Why this scenario?** A travel concierge touches three challenges every agent developer hits:

- **Multi-step reasoning** — break *"plan my trip"* into smaller, scoped jobs.
- **Policy compliance** — never recommend something that breaks the company's rules.
- **Quality vs. cost trade-offs** — pick a model fast enough for users *and* smart enough for nuance.

It is the same shape as a refunds bot, an HR helper, a procurement assistant — anything that talks
to people, follows rules, and reaches into multiple back-end systems.

## 2. The big picture

Here is everything that has to be in place for the concierge to work, and where each piece lives:

```mermaid
flowchart TB
    subgraph foundry["🏭 Microsoft Foundry (cloud)"]
        subgraph project["📋 Foundry project"]
            cmd["☁️ command-a<br/>chat + reasoning"]
            emb["☁️ embed-v-4-0<br/>semantic search"]
            rer["☁️ Cohere-rerank-v4.0-pro<br/>relevance scoring"]
        end
        appins["📷 Application Insights<br/>traces + monitoring"]
    end
    subgraph local["💻 Your machine (or Codespace)"]
        maf["🛠️ MAF agents<br/>concierge + specialists"]
        evals["📊 azure-ai-evaluation<br/>quality + safety scores"]
        redteam["🛡️ RedTeam scanner<br/>safety probes"]
    end
    maf -->|chat| cmd
    maf -->|embed| emb
    maf -->|rerank| rer
    maf -->|spans| appins
    evals -->|judge model| cmd
    redteam --> maf
```

**Two halves, one workshop.** The **cloud half** (Foundry) holds the models and the observability
backbone. The **local half** (MAF on your laptop) holds the agent code, the evaluators, and the
red-team scanner. You ship the cloud half **once** in Lab 0 and iterate on the local half throughout
Lab 1 — fast feedback, no portal click-ops.

## 3. What is Microsoft Foundry?

**Microsoft Foundry** is where you deploy and manage AI models. Think of it as an
*app-store-plus-control-panel* for foundation models from many providers — all behind one billing
relationship, one identity model, and one place to look when something goes wrong.

**The pieces you will use:**

| Concept | What it is | Mental model |
|---|---|---|
| **Account** | Top-level container; usually one per business unit | The building |
| **Project** | Workspace inside the account that groups deployments, agents, evals, and monitoring | A floor in the building |
| **Deployment** | A chosen model made callable at a chosen capacity (e.g., `command-a` at 100 RPM) | An installed phone line you can dial |
| **Application Insights** | Captures every call so traces, latency, and errors are visible without writing logging code | The CCTV camera over each room |

**Why this matters for you as a developer.** Instead of stitching together several vendor SDKs,
separate monitoring tools, and separate governance dashboards, you get one endpoint, one identity
model (Microsoft Entra), and a consistent set of tools no matter which model you pick. The same
code works for a Cohere model today and an OpenAI model tomorrow — Foundry handles the plumbing.

## 4. Why Cohere models?

Foundry hosts many model families. This workshop focuses on **Cohere** because Cohere ships a
complementary trio that together covers most enterprise agent workloads:

| Cohere model | What it does | When you reach for it |
|---|---|---|
| **`command-a`** | Chat, reasoning, tool use | The brain of an agent — strong instruction following, citation-friendly, multilingual. |
| **`embed-v-4-0`** | Turns text or images into numeric vectors | Find documents by *meaning*, not keyword match — the foundation of RAG and similar-item search. |
| **`Cohere-rerank-v4.0-pro`** | Re-orders a candidate list by true relevance | Take a noisy first-pass result and surface the genuinely useful items before the chat model sees them. |

**The travel-desk analogy.** Picture a three-person research team behind the concierge counter:

- **embed** is the *librarian* who knows where every travel document, review, and policy section lives.
- **rerank** is the *editor* who reads the librarian's shortlist and reorders it by what the traveler actually needs.
- **command-a** is the *senior agent* who reads the top picks and writes the polished trip plan back to the customer.

You meet **command-a** in Lab 1 and **embed + rerank** in Lab 2.

## 5. What is the Microsoft Agent Framework (MAF)?

**MAF** is Microsoft's open-source Python library for building AI agents in your own code.
It is a small, focused toolkit — `Agent`, tools, chat clients, OpenTelemetry tracing — that
you `pip install` like any other package.

**Foundry gives you two equally valid ways to ship an agent:**

```mermaid
flowchart LR
    foundry["🏭 Foundry-hosted agent<br/>(Agent Service)"]
    maf["🛠️ MAF agent<br/>(open source, your code)"]
    cohere["☁️ Same Cohere deployment<br/>in your Foundry project"]
    foundry -->|chat| cohere
    maf -->|chat| cohere
```

- **Hosted Foundry Agent Service** — fastest path when your model is on the supported list and you
  want zero infrastructure. Foundry stores the agent definition, runs it, and shows it in the portal.
- **MAF + Foundry deployments** — lets you build with an open-source framework using models
  deployed in Foundry. You keep Foundry's observability and security model and gain the freedom to
  use any model that has a Foundry endpoint (including Cohere) plus any orchestration pattern you
  can express in Python.

The workshop uses **MAF** so you experience the *full* developer loop — write code, read traces,
run evals, scan for safety — against a real Cohere deployment in Foundry.

## 6. Step 1 — Stand up the factory (Lab 0)

Before any agent code runs, you need a Foundry project with the right deployments and observability.
Lab 0 automates this end-to-end.

```mermaid
flowchart LR
    login["az login"]
    provision["provision.sh<br/>(Bicep)"]
    setup["setup-env.sh<br/>(writes .env)"]
    verify["verify.sh<br/>(pings every model)"]
    login --> provision --> setup --> verify
```

**What Lab 0 creates:**

- A Microsoft Foundry account in `eastus2` (one region that hosts all four models).
- A project workspace inside that account.
- Four model deployments at **100 RPM** capacity each: `command-a`, `embed-v-4-0`,
  `Cohere-rerank-v4.0-pro`, `text-embedding-3-small`.
- Application Insights + Log Analytics, so tracing works the moment you start Lab 1.
- A populated `cohere/.env` so every notebook in the workshop can find its endpoints.

**Why this matters.** You only set up the cloud half *once*. Everything else is local Python you can
edit, rerun, and iterate on freely. **GitHub Codespaces is the recommended starting point** because
the devcontainer installs every dependency on first open — no local Python wrangling.

> 💡 **Capacity gotcha.** Portal-clicked deployments default to capacity `1` (≈1 request per minute),
> which throttles Lab 1's evaluation loops the moment they start. The Lab 0 scripts pin every
> deployment at `100` capacity so the evals run smoothly.

## 7. Step 2 — Build the concierge (Lab 1, notebooks 01 → 03)

With the factory ready, you assemble the agent in three short steps:

1. **`01-verify-cohere.ipynb`** — confirm MAF can chat with your `command-a` deployment. One round
   trip. If this fails, fix it before going further.
2. **`02-build-multi-agent.ipynb`** — wire up the concierge orchestrator and three specialist
   agents (flight, hotel, car) using the *agents-as-tools* pattern. Each specialist owns its own
   booking tools and JSON catalog.
3. **`03-trace-multi-agent.ipynb`** — turn on OpenTelemetry. Every agent turn, specialist hop, and
   tool call becomes a span you can read in the terminal or in the Foundry Monitoring tab.

**The agents-as-tools pattern in 30 seconds:**

```mermaid
flowchart LR
    concierge["🧳 Concierge"]
    flight["✈️ Flight agent<br/>(exposed as one tool)"]
    search["search_flights()"]
    book["book_flight()"]
    concierge -->|flight_agent(query)| flight
    flight --> search
    flight --> book
```

The concierge sees the specialist as a **single tool with a single signature**. It does not need
to know what tools the specialist owns — the specialist decides. This keeps the concierge prompt
short and lets you swap specialists without rewriting the orchestrator.

**Why trace at this stage?** Because the next step is to measure quality, and you cannot improve
what you cannot see. Traces show you exactly which specialist a request reached, what arguments
it sent to each tool, and how long every hop took.

## 8. Step 3 — Measure quality (Lab 1, notebook 04, rounds 1 → 4)

This is the heart of the workshop: **four progressively smarter versions of the same agent,
scored on the same 20-prompt dataset.** Same dataset + same evaluators = honest before/after
comparison.

```mermaid
flowchart LR
    r1["Round 1<br/>Baseline<br/>generic prompt"]
    r2["Round 2<br/>Grounded<br/>+ policy + catalog"]
    r3["Round 3<br/>Policy judge<br/>+ custom LLM-as-judge"]
    r4["Round 4<br/>Multi-agent<br/>+ specialists + tools"]
    r1 --> r2 --> r3 --> r4
```

**What an "evaluator" actually is.** A small program that scores a single agent response. The
built-in panel in `azure-ai-evaluation` includes:

- **Relevance** — did the answer actually match the question?
- **Coherence** — does it read smoothly?
- **Fluency** — is the language natural?
- **Groundedness** — are claims supported by context you provided?
- **Intent resolution** — did the agent address the real underlying intent?
- **Task adherence** — did it actually complete what was asked, end-to-end?

Most of these use **LLM-as-judge** — the evaluator sends the agent's answer plus the original prompt
to a model (here, `command-a` itself) and asks it to score against a defined rubric.

**Why four rounds, not one?**

| Round | Hypothesis tested | Typical takeaway |
|---|---|---|
| 1 — Baseline | *"How good is the model out of the box?"* | Sets the floor scores. |
| 2 — Grounded | *"Does putting policy + catalog into the prompt help?"* | Grounding usually lifts relevance + groundedness sharply. |
| 3 — Policy judge | *"The built-ins miss policy violations — add a custom judge."* | Catches what generic evaluators cannot. |
| 4 — Multi-agent | *"Do specialists + tools beat a single-prompt agent?"* | Tools let the agent retrieve real data, usually the biggest lift. |

This is **model optimization without retraining**. You change instructions, context, and
orchestration — not weights — and measure the lift each change buys you. Most production agent
teams live in this loop for weeks before they ever consider fine-tuning.

## 9. Step 4 — Stress-test for safety (Lab 1, notebook 05)

Evaluators measure *quality*. **Red-teaming** measures *safety* — does the agent refuse harmful
requests, decline obvious jailbreaks, and avoid leaking data?

```mermaid
flowchart LR
    scanner["🛡️ RedTeam scanner<br/>generates adversarial prompts"]
    concierge["🧳 Your local concierge"]
    judge["⚖️ Safety judge<br/>scores each response"]
    report["📋 Risk report<br/>uploaded to Foundry"]
    scanner --> concierge --> judge --> report
```

`azure.ai.evaluation.red_team.RedTeam` ships with probe strategies for risk categories such as
**violence**, **self-harm**, **hate / unfairness**, **protected material**, and **prohibited
actions**. The scanner sends those probes to your local concierge, judges the answers, and
uploads a risk report to your Foundry project so the whole team can review it.

**Why run this locally?** Because the scanner needs a callable target it can probe directly. Running
it in-process against your MAF agent gives the fastest feedback loop — you spot a safety regression
the moment you change instructions, long before it could reach a real user.

## 10. Step 5 — Specialize with Embed and Rerank (Lab 2)

Lab 1 used **command-a** for everything because chat is the most familiar capability. Lab 2 zooms
in on the other two Cohere models so you can pick the right tool for each job in your own apps.

```mermaid
flowchart LR
    raw["📄 Raw documents<br/>(flights, hotels, reviews)"]
    embed["embed-v-4-0<br/>(librarian)"]
    vectors["🗂️ Vector index"]
    query["❓ User query"]
    candidates["🔎 Top 20 candidates"]
    rerank["Cohere-rerank-v4.0-pro<br/>(editor)"]
    final["✅ Top 3"]
    chat["command-a<br/>(senior agent)"]
    answer["💬 Final reply"]
    raw --> embed --> vectors
    query --> vectors --> candidates --> rerank --> final --> chat --> answer
```

**When to use which:**

- **embed** when you have *many* documents and need to find a small set that's semantically
  related to a query (RAG, similar-item search).
- **rerank** when your first-pass retrieval is noisy and you need to surface the genuinely useful
  items before sending them to the chat model.
- **command-a** when you need to *reason* over the picked items and write the response.

The capstone notebook (`optional-try-it-yourself-travel.ipynb`) re-uses the travel scenario so you
connect Lab 2 patterns back to the Lab 1 concierge.

## 11. The big idea — build, measure, improve

If you take away one thing from this workshop, take away this loop:

```mermaid
flowchart LR
    build["1. Build<br/>(MAF agents)"]
    trace["2. Trace<br/>(OpenTelemetry)"]
    eval["3. Evaluate<br/>(quality scores)"]
    redteam["4. Red-team<br/>(safety scores)"]
    improve["5. Improve<br/>(prompt / tools / model)"]
    build --> trace --> eval --> redteam --> improve --> build
```

**Why a loop, not a checklist?** Because the easy part of agent engineering is the first working
prototype. The hard part is knowing whether your next change made things better or worse.
Evaluations give you a number. Red-team scans give you a safety floor. Tracing tells you *why* a
number moved. The loop turns vibes into evidence — that is what makes an agent production-ready.

**Cohere shows up at three different points in this loop:**

1. **Drives** the agent — `command-a` reasons and writes the response.
2. **Judges** the agent — `command-a` again, this time as the LLM-as-judge inside evaluators.
3. **Grounds** the agent — `embed-v-4-0` + `Cohere-rerank-v4.0-pro` find and rank the context the
   agent reasons over.

That is why the workshop lives at the intersection of **Foundry + MAF + Cohere** — each does one
job exceptionally well, and together they form a complete development loop without leaving the
Microsoft ecosystem.

## 12. Optional — am I ready to start?

Run the cell below to confirm your environment is wired up. If you see ✅, head to
[Lab 1](./lab-1-foundry-maf/README.md). If anything is missing, head back to
[Lab 0](./lab-0-setup/SETUP.md) and re-run `setup-env.sh`.

In [ ]:
"""Optional readiness check — looks for the env vars Lab 1 needs."""
import os
from pathlib import Path

from dotenv import load_dotenv

# Find the repo root without hardcoding the workshop folder name (forks may rename it)
repo_root = Path.cwd().resolve()
while not (repo_root / '.git').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
env_path = repo_root / 'cohere' / '.env'
load_dotenv(env_path)

required = [
    'AZURE_AI_ENDPOINT',
    'FOUNDRY_PROJECT_ENDPOINT',
    'COMMAND_A_DEPLOYMENT',
    'EMBED_V4_DEPLOYMENT',
    'RERANK_DEPLOYMENT',
]
missing = [k for k in required if not os.getenv(k)]

if missing:
    print('⚠️  Missing env vars — run Lab 0 first:')
    for k in missing:
        print(f'   - {k}')
    print(f'\n   Looked for .env at: {env_path}')
else:
    print('✅ Environment looks good. You are ready for Lab 1.')
    print(f'   Endpoint:   {os.getenv("AZURE_AI_ENDPOINT")}')
    print(f'   Project:    {os.getenv("FOUNDRY_PROJECT_ENDPOINT")}')
    print(f'   Deployments: {os.getenv("COMMAND_A_DEPLOYMENT")}, '
          f'{os.getenv("EMBED_V4_DEPLOYMENT")}, '
          f'{os.getenv("RERANK_DEPLOYMENT")}')

## 13. Where to go next

| If you... | Open |
|---|---|
| Have **never run the workshop** | [`lab-0-setup/SETUP.md`](./lab-0-setup/SETUP.md) — provision Foundry + Cohere deployments |
| Have Lab 0 done and want to **build the concierge** | [`lab-1-foundry-maf/README.md`](./lab-1-foundry-maf/README.md) |
| Have Lab 1 done and want to **explore Cohere capabilities** | [`lab-2-cohere-capabilities/README.md`](./lab-2-cohere-capabilities/README.md) |
| Want the **workshop overview** | [`README.md`](./README.md) |

**Further reading on the technologies in this workshop:**

- [Microsoft Foundry documentation](https://learn.microsoft.com/azure/ai-foundry/)
- [Microsoft Agent Framework on GitHub](https://github.com/microsoft/agent-framework)
- [Cohere on Microsoft Foundry — model catalog](https://learn.microsoft.com/azure/ai-foundry/model-inference/concepts/models)
- [`azure-ai-evaluation` SDK reference](https://learn.microsoft.com/python/api/overview/azure/ai-evaluation-readme)

Happy building. 🚀